# Lesson 7.3: Rank-Based Tests: When Averages Mislead

Show how rank-based tests (Mann-Whitney, Wilcoxon, Kruskal-Wallis) cope with an outlier that a t-test cannot, using the lesson's app-timing, pain and teaching data.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A designer wants to know whether a redesigned app helps people finish a task faster. Six people used the new design and six the old one; times in seconds are in the lesson. One old-design user took 95 seconds, maybe after a phone call. Does the new design really work faster, and how do we test that when one wild value can fool the usual t-test?

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: Two groups: the t-test versus ranks

Task times in seconds for a new and an old app design. The old group has one 95-second outlier. Welch's t-test uses the size of every value, so the outlier hides the pattern. Mann-Whitney only uses the order of the values.

In [1]:
import numpy as np  # Load NumPy, the library for numbers.
from scipy import stats  # Load SciPy's statistics tools, which contain both the t-test and the rank tests.

new = [12, 15, 11, 14, 19, 13]  # Task times in seconds for the six people on the new design. None is extreme.
old = [18, 22, 25, 17, 30, 95]  # Task times for the six people on the old design. The 95 is the outlier.

t = stats.ttest_ind(new, old, equal_var=False)  # Welch's t-test compares the two means. equal_var=False is the safe version that does not assume equal spreads.
mw = stats.mannwhitneyu(new, old, alternative='two-sided', method='exact')  # Mann-Whitney compares the ranks instead of the sizes. method='exact' works out the exact p-value, best for small samples.

print(f'Welch t-test : t = {t.statistic:.2f}, p = {t.pvalue:.3f}')  # Print the t statistic and p-value. Welch uses every value's size, so the 95 inflates the spread.
print(f'Mann-Whitney : U = {mw.statistic:.0f}, p = {mw.pvalue:.4f}')  # Print U (a count of pair wins, explained next) and its p-value. Only the order of values matters here.

Welch t-test : t = -1.67, p = 0.156
Mann-Whitney : U = 2, p = 0.0087


**What this means.** Welch's t-test gives t = -1.67, p = 0.156: not significant, because the 95-second outlier makes the old group's spread huge. Mann-Whitney only uses the order of times and gives U = 2, p = 0.0087, which is significant. The new design is faster, and ranks can see it.

## Step 2: How strong is the Mann-Whitney effect?

U counts in how many of the 36 new-versus-old pairs the new design had the larger (slower) time. Here that is only 2, so almost every pair says the new design was faster. The rank-biserial correlation turns this into a size from 0 to 1.

In [2]:
n1, n2 = len(new), len(old)  # Count the people in each group: 6 and 6.
u = mw.statistic  # U is the number the test reports: in how many pairs the new time came out larger than the old.

r = 1 - 2 * u / (n1 * n2)  # Rank-biserial r = 1 - 2U / (n1 x n2), turning U into a size from -1 to +1. It is the Mann-Whitney effect size.

print(f'new time larger in {u:.0f} of {n1 * n2} pairs')  # Print how many of the 36 new-versus-old pairs had the new design slower.
print(f'old time larger in {n1 * n2 - u:.0f} of {n1 * n2} pairs')  # The other pairs are the ones where the old design was slower. The two counts add to 36.
print(f'rank-biserial r = {r:.2f}')  # Print the effect size, rounded to 2 decimals.

new time larger in 2 of 36 pairs
old time larger in 34 of 36 pairs
rank-biserial r = 0.89


**What this means.** U = 2 means the new design was slower in only 2 of the 36 pairs; the old design was slower in 34. The rank-biserial correlation is 0.89, close to its maximum of 1, so almost every new time beats almost every old time. That is a large effect.

## Step 3: Paired data: Wilcoxon signed-rank

Pain scores (0 to 10) for eight patients before and after treatment. The same people are measured twice, so we test the differences. The Wilcoxon statistic is the smaller of the two rank sums (positive and negative differences).

In [3]:
before = [7, 8, 6, 9, 7, 6, 8, 7]  # Pain scores (0 to 10) for eight patients before treatment. The same eight people will be measured again.
after = [3, 1, 4, 0, 4, 7, 2, 2]  # Pain scores for the same eight people after treatment, in the same order.

w = stats.wilcoxon(before, after)  # Wilcoxon signed-rank test for paired data. It ranks the size of each person's change and checks whether the ups and downs balance.
pt = stats.ttest_rel(before, after)  # The paired t-test on the same data, for comparison. It also works on each person's difference.

print(f'Wilcoxon : W = {w.statistic:.0f}, p = {w.pvalue:.4f}')  # Print Wilcoxon's W, the smaller of the two rank sums, with its p-value.
print(f'Paired t : t = {pt.statistic:.2f}, p = {pt.pvalue:.4f}')  # Print the paired t statistic and p-value.

Wilcoxon : W = 1, p = 0.0156
Paired t : t = 3.97, p = 0.0054


**What this means.** The Wilcoxon test gives W = 1, p = 0.0156: pain scores really dropped, because nearly every patient improved (only patient 6 got worse, by a point). The paired t-test gives t = 3.97, p = 0.0054 and agrees here, since there is no extreme outlier.

## Step 4: Three groups: Kruskal-Wallis

Three teaching methods with three students each. Kruskal-Wallis is the rank version of one-way ANOVA. With groups this small the p-value is an approximation, so treat it as a guide.

In [4]:
a = [65, 72, 68]  # Test scores of three students taught by method a.
b = [80, 85, 78]  # Test scores of three students taught by method b.
c = [55, 60, 58]  # Test scores of three students taught by method c.

kw = stats.kruskal(a, b, c)  # Kruskal-Wallis is the rank version of one-way ANOVA. It ranks all nine scores together and compares the groups' rank totals.
f = stats.f_oneway(a, b, c)  # Run ordinary one-way ANOVA on the same numbers, for comparison.

print(f'Kruskal-Wallis: H = {kw.statistic:.2f}, p = {kw.pvalue:.4f}')  # Print H, Kruskal-Wallis's statistic, and its p-value (an approximation with groups this small).
print(f'ANOVA         : F = {f.statistic:.2f}, p = {f.pvalue:.4f}')  # Print ANOVA's F and p-value.

Kruskal-Wallis: H = 7.20, p = 0.0273
ANOVA         : F = 38.78, p = 0.0004


**What this means.** Kruskal-Wallis gives H = 7.20, p = 0.0273, so the three methods differ in rank. ANOVA gives F = 38.78, p = 0.0004. With only 3 students per group both are rough guides, but they point the same way: the methods differ.

## So, what is the answer?

Yes, the new design is faster. The Mann-Whitney test gives p = 0.0087 and the new design was quicker in 34 of 36 pairs, even though the Welch t-test (p = 0.156) was fooled by the 95-second outlier. Rank tests resist outliers, but with only six people per design this is a small study, and we should find out why one person took 95 seconds.

## Now you try

A food-delivery firm compares delivery times (minutes) from two couriers' routes: a fast route (7 deliveries) and a slow route (7 deliveries). One slow-route delivery took 190 minutes because of a vehicle breakdown. Is the slow route really slower?

**Data:** fast = [32, 35, 30, 38, 33, 36, 31]
slow = [41, 45, 39, 48, 44, 42, 190]

1. Run Welch's t-test and report t and p.
2. Run an exact two-sided Mann-Whitney test and report U and p.
3. Count in how many of the 49 pairs the fast route was slower, and give the rank-biserial correlation.
4. Which test do you trust here, and why?

*Hint:* Use stats.ttest_ind(..., equal_var=False) and stats.mannwhitneyu(..., method='exact'). The effect size is 1 - 2U / (n1 x n2).

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
import numpy as np
from scipy import stats

fast = [32, 35, 30, 38, 33, 36, 31]
slow = [41, 45, 39, 48, 44, 42, 190]
t = stats.ttest_ind(fast, slow, equal_var=False)
mw = stats.mannwhitneyu(fast, slow, alternative='two-sided', method='exact')
print(f'Welch t = {t.statistic:.2f}, p = {t.pvalue:.3f}')
print(f'Mann-Whitney U = {mw.statistic:.0f}, p = {mw.pvalue:.5f}')
n = len(fast) * len(slow)
print(f'pairs where fast was larger: {mw.statistic:.0f} of {n}')
print(f'rank-biserial r = {1 - 2 * mw.statistic / n:.2f}')
print('means', np.mean(fast).round(1), np.mean(slow).round(1))
print('medians', np.median(fast), np.median(slow))

Welch t = -1.45, p = 0.196
Mann-Whitney U = 0, p = 0.00058
pairs where fast was larger: 0 of 49
rank-biserial r = 1.00
means 33.6 64.1
medians 33.0 44.0


**Answer.** Welch's t-test gives t = -1.45, p = 0.196, not significant, because the 190-minute breakdown drags the slow mean to 64.1 and inflates its spread. Mann-Whitney gives U = 0, p = 0.00058: the fast route was slower in 0 of 49 pairs, so the rank-biserial r = 1.00 (as strong as it gets). Medians are 33 and 44. Trust the rank test here, since the outlier distorts the t-test.

## The same question in R, SPSS and Stata

**R**

```r
new <- c(12, 15, 11, 14, 19, 13)  # Task times for the new design.
old <- c(18, 22, 25, 17, 30, 95)  # Task times for the old design, including the 95-second outlier.
t.test(new, old)  # t.test compares the means; by default R runs Welch's version, which does not assume equal spreads.
wilcox.test(new, old)  # wilcox.test on two samples is the Mann-Whitney test, which compares ranks.
before <- c(7, 8, 6, 9, 7, 6, 8, 7)  # Pain scores before treatment for eight patients.
after <- c(3, 1, 4, 0, 4, 7, 2, 2)  # Pain scores for the same eight patients after treatment, in the same order.
wilcox.test(before, after, paired = TRUE)  # paired = TRUE makes wilcox.test the Wilcoxon signed-rank test on each patient's before-after difference.
score <- c(65, 72, 68, 80, 85, 78, 55, 60, 58)  # Scores of nine students, three in each of three teaching methods.
method <- factor(rep(c("A", "B", "C"), each = 3))  # A factor naming each student's method: A, B or C, three times each.
kruskal.test(score ~ method)  # kruskal.test is the Kruskal-Wallis test of score by method.
```

How to read it: Welch: t = -1.6654, p-value = 0.1557 (not significant). Mann-Whitney: W = 2, p-value = 0.008658 (significant). Paired Wilcoxon: V = 35, p-value = 0.01562. Kruskal-Wallis: chi-squared = 7.2, df = 2, p-value = 0.02732. The rank tests find what the outlier hides from the t-test.

**SPSS** (Analyze > Nonparametric Tests > Legacy Dialogs > 2 Independent Samples (Mann-Whitney U) and 2 Related Samples (Wilcoxon))

```
DATA LIST FREE / time group.
BEGIN DATA
12 1  15 1  11 1  14 1  19 1  13 1  18 2  22 2  25 2  17 2  30 2  95 2
END DATA.
NPAR TESTS /M-W=time BY group(1,2).
DATA LIST FREE / before after.
BEGIN DATA
7 3  8 1  6 4  9 0  7 4  6 7  8 2  7 2
END DATA.
NPAR TESTS /WILCOXON=before WITH after (PAIRED).
```

How to read it: Mann-Whitney: look at Test Statistics, Mann-Whitney U = 2.000 and Asymp. Sig. (2-tailed) about 0.010 (exact 0.0087). Wilcoxon: the Ranks table shows rank sums of 35 and 1, and Asymp. Sig. is about 0.02 (exact 0.0156). Both are below 0.05, so both tests find a real difference.

**Stata**

```
clear
set obs 12
generate time = real(word("12 15 11 14 19 13 18 22 25 17 30 95", _n))
generate group = _n > 6
ranksum time, by(group)
ttest time, by(group) unequal
generate before = real(word("7 8 6 9 7 6 8 7", _n))
generate after = real(word("3 1 4 0 4 7 2 2", _n))
signrank before = after
generate score = real(word("65 72 68 80 85 78 55 60 58", _n))
generate method = ceil(_n / 3) if _n <= 9
kwallis score, by(method)
```

How to read it: ranksum prints a z statistic and p-value (about 0.010, with an exact value near 0.0087 shown too) for the rank test; ttest prints Pr(|T| > |t|) around 0.156. signrank shows sum ranks 35 and 1 with p about 0.02. kwallis shows chi-squared = 7.200 and Prob = 0.0273.